In [141]:
import pandas as pd
df = pd.read_csv('../data/raw/tech_layoffs_til_2025.csv')

Standardize "United Kingdom" to "UK" since UK is the more common spelling 
in this dataset (74 vs 5 occurrences).

In [142]:
df['Country'] = df['Country'].replace('United Kingdom', 'UK')

Industry column has 118 categories; many appear only 1-2 times. 
Grouping low-frequency categories (appearing fewer than 5 times) into 
"Other" to make industry-level analysis more meaningful.

In [143]:
industry_counts = df['Industry'].value_counts()
rare_industries = industry_counts[industry_counts < 5].index
df['Industry'] = df['Industry'].apply(lambda x: 'Other' if x in rare_industries else x)

In [144]:
df['Industry'].value_counts()

Industry
Finance                          289
Other                            228
Retail                           176
Healthcare                       156
Transportation                   142
Consumer                         134
Food                             133
Marketing                        121
HR                                79
Security                          79
Education                         77
Real Estate                       74
Media                             74
Data                              66
Crypto                            65
Travel                            52
Software Development              44
Logistics                         44
Sales                             41
Infrastructure                    37
Energy                            32
Product                           30
Hardware                          29
Support                           29
Fitness                           22
AI                                17
Gaming                       

In [145]:
df['Industry'] = df['Industry'].replace({'e-commerce': 'E-commerce'})

In [146]:
industry_lower = df['Industry'].str.strip().str.lower()
duplicate_check = df.groupby(industry_lower)['Industry'].nunique()
result = duplicate_check[duplicate_check > 1]
print(result)

Series([], Name: Industry, dtype: int64)


"Unknown" in the Stage column (356 records) carries no usable funding-stage 
information. Converting it to a proper missing value (NaN) so it's handled 
consistently with other missing data during analysis.

In [147]:
import numpy as np
df['Stage'] = df['Stage'].replace('Unknown', np.nan)

In [148]:
# To verify if there's still 'Unknown' in 'Stage' column
df['Stage'].value_counts()

Stage
Post-IPO          574
Series B          266
Series C          211
Series D          203
Acquired          176
Series E          123
Series A          116
Series F           66
Seed               49
Private Equity     37
Series H           27
Series G           19
Subsidiary         11
Series I            8
Series J            6
Name: count, dtype: int64

Region column is too coarse for analysis (67% classified as "other") and 
is being dropped. Country will be used instead for geographic analysis.

In [149]:
df = df.drop(columns = ['Region'])

Binning Company_Size_before_Layoffs into 3 categories based on the quantile 
analysis from the exploration phase: Small (≤300), Mid-size (301-1000), 
Large (>1000).

In [150]:
df['Company_Size_Category'] = pd.cut(
    df['Company_Size_before_Layoffs'],
    bins=[0, 300, 1000, 99999999999],
    labels=['Small', 'Mid-size', 'Large']
)

In [151]:
df['Company_Size_Category'].value_counts()

Company_Size_Category
Small       587
Mid-size    585
Large       584
Name: count, dtype: int64

Dropping `Nr` (a row index with no analytical value) and `Continent` 
(too coarse, overlaps with Country which is already used for geographic 
analysis).

In [152]:
df = df.drop(columns=['Nr', 'Continent'])

In [153]:
def check_case_duplicates(df, col):
    """
    Check a column for duplicate categories caused by 
    inconsistent casing or leading/trailing spaces.
    Returns categories with more than one original spelling 
    (empty Series if clean).
    """
    col_lower = df[col].str.strip().str.lower()
    dup_check = df.groupby(col_lower)[col].nunique()
    result = dup_check[dup_check > 1]
    return result

In [154]:
categorical_cols = df.select_dtypes(include='object').columns

for col in categorical_cols:
    result = check_case_duplicates(df, col)
    if len(result) > 0:
        print(f"Column '{col}' has duplicates:")
        print(result)
        print()

Column 'Company' has duplicates:
Company
7shifts        2
bytedance      2
clearco        2
freshbooks     2
gameskraft     2
gupshup        2
jamf           2
tripadvisor    2
uipath         2
wayfair        2
Name: Company, dtype: int64



In [155]:
dup_companies = check_case_duplicates(df, 'Company')

for name in dup_companies.index:
    variants = df[df['Company'].str.strip().str.lower() == name]['Company'].unique()
    print(name, "->", variants)

7shifts -> ['7shifts' '7Shifts']
bytedance -> ['ByteDance' 'Bytedance']
clearco -> ['Clearco' 'ClearCo']
freshbooks -> ['Freshbooks' 'FreshBooks']
gameskraft -> ['Gameskraft' 'GamesKraft']
gupshup -> ['GupShup' 'Gupshup']
jamf -> ['Jamf' 'JAMF']
tripadvisor -> ['TripAdvisor' 'Tripadvisor']
uipath -> ['UiPath' 'UIPath']
wayfair -> ['Wayfair' 'Wayfair ']


In [156]:
df[df['Company'].str.strip().str.lower() == 'clearco'][['Company', 'Industry', 'Country', 'Stage']]

,Company,Industry,Country,Stage
602,Clearco,Finance,Canada,Series C
960,ClearCo,Finance,Canada,Series C


**Company Name Inconsistency Check**

While building the Tableau dashboard, I noticed the Industry column had a 
duplicate category caused by inconsistent casing (`e-commerce` vs `E-commerce`). 
That made me wonder if the same issue exists in other text columns — especially 
Company, since company names are entered manually and prone to typos or 
inconsistent capitalization.

Instead of checking column by column manually, I wrote a small reusable function, 
`check_case_duplicates()`, that flags any category appearing under more than one 
casing/spacing variant, then looped it across all categorical columns at once.

The scan came back clean except for one column: **Company**. 11 companies had 
two different casing variants each (e.g. `bytedance` / `ByteDance`, `uipath` / 
`UiPath`).

One case needed extra attention: `Clearco` vs `ClearCo`. These aren't just a 
casing difference — they could be two unrelated companies that happen to share 
a similar name (Clearco, formerly Clearbanc, a Canadian fintech, vs ClearCo, 
formerly ClearCompany, a US HR software company). I checked both rows' Industry, 
Country, and Stage, and both matched Clearco's profile (Finance / Canada / 
Series C), confirming they're the same company and safe to merge.

For the rest, I manually verified each company's official brand spelling 
(e.g. Tripadvisor dropped the capital "A" in a 2020 rebrand — easy to get wrong 
if you go by instinct) before building the replacement dictionary below.

In [157]:
company_name_fixes = {
    '7Shifts': '7shifts',
    'Bytedance': 'ByteDance',
    'ClearCo': 'Clearco',
    'Freshbooks': 'FreshBooks',
    'GameSkraft': 'Gameskraft',
    'GupShup': 'Gupshup',
    'JAMF': 'Jamf',
    'TripAdvisor': 'Tripadvisor',
    'UIPath': 'UiPath',
    'Wayfair ': 'Wayfair',
}

df['Company'] = df['Company'].replace(company_name_fixes)

In [158]:
df.to_csv('../data/cleaned/tech_layoffs_cleaned.csv', index=False)

In [ ]:
df[df['Industry'].str.strip().str.lower() == 'e-commerce']['Industry'].unique()

array(['E-commerce'], dtype=object)

: 